In [ ]:
import pandas as pd
from skrub import TableReport

In [ ]:
df = pd.read_csv('../notebooks/data_grappling_all.csv')

In [ ]:
rapport=TableReport(df)
rapport

In [ ]:
df["id_combattant"] = df["fighter_url"].factorize()[0] + 1
df = df.drop(columns= "fighter_url")
df

In [ ]:
defaites = df[df["Résultat"] == "L"]
defaites['Type_victoire'].value_counts()

In [ ]:
def categoriser(type_vic):
    if pd.isna(type_vic):
        return 'Inconnu'
    t = type_vic.lower()
    
    if 'pts' in t or 'points' in t or 'adv' in t:
        return 'Points/Avantages'
    elif 'referee' in t or 'decision' in t:
        return 'Décision arbitre'
    elif 'choke' in t or 'strangle' in t or 'strangulation' in t:
        return 'Étranglement'
    elif 'heel hook' in t or 'kneebar' in t or 'toe hold' in t or 'leg lock' in t:
        return 'Lock jambe'
    elif 'armbar' in t or 'kimura' in t or 'americana' in t or 'omoplata' in t:
        return 'Lock bras/épaule'
    elif 'triangle' in t:
        return 'Triangle'
    else:
        return 'Autre soumission'

defaites['categorie'] = defaites['Type_victoire'].apply(categoriser)
print(defaites['categorie'].value_counts())

In [ ]:
print(defaites[defaites['categorie'] == 'Inconnu']['Type_victoire'].value_counts())

correspond donc aux donnees manquantes

In [ ]:
import matplotlib.pyplot as plt

# Supprimer les inconnus
defaites_clean = defaites[defaites['categorie'] != 'Inconnu']

freq = defaites_clean['categorie'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if i == 0 else '#3498db' if i == 1 else '#e74c3c' 
          for i in range(len(freq))]

bars = ax.barh(freq.index, freq.values, color='steelblue', edgecolor='white')

# Ajouter les valeurs sur les barres
for bar, val in zip(bars, freq.values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{val} ({val/len(defaites_clean)*100:.1f}%)',
            va='center', fontsize=10)

ax.set_title('Comment les combattants BJJ perdent-ils ?', fontsize=14, fontweight='bold')
ax.set_xlabel('Nombre de défaites')
ax.set_xlim(0, freq.values.max() * 1.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

La grande majorité des défaites en BJJ compétitif se font aux points ou aux avantages (~55%) et aux décisions arbitrales (~13%). Les soumissions pures ne représentent qu'environ 30% des défaites au total, ce qui montre que le BJJ de compétition moderne est devenu un sport très tactique où gagner aux points est la stratégie dominant

In [ ]:
mapping = defaites_clean.groupby("id_combattant")["last_name"].first()
# Filtrer uniquement les soumissions
soumissions = ['Étranglement', 'Lock bras/épaule', 'Lock jambe', 'Autre soumission']
defaites_soum = defaites_clean[defaites_clean['categorie'].isin(soumissions)]

# Compter par combattant
par_combattant = defaites_soum.groupby("id_combattant").size().sort_values(ascending=False)

par_combattant = par_combattant.to_frame("nb_defaites")
par_combattant["last_name"] = par_combattant.index.map(mapping)

print(par_combattant.head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

top20 = par_combattant.head(20)

bars = ax.barh(top20['last_name'][::-1], top20['nb_defaites'][::-1], color='steelblue', edgecolor='white')

for bar, val in zip(bars, top20['nb_defaites'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

ax.set_title('Top 20 combattants avec le plus de défaites par soumission', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Nombre de défaites par soumission')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, top20['nb_defaites'].max() * 1.15)

plt.tight_layout()
plt.show()

Le classement brut est biaisé par le volume — Sousa et Hernandez apparaissent en tête simplement parce qu'ils ont fait beaucoup de combats. Ce n'est pas un indicateur de vulnérabilité réelle.

c'est peu represnetatif ; regardons plutot le ratio

In [ ]:
# Total combats par combattant
total_combats = df.groupby('id_combattant').size()

# Défaites par soumission par combattant
def_soum = defaites_soum.groupby('id_combattant').size()

# Ratio
ratio_soum = (def_soum / total_combats * 100).dropna().sort_values(ascending=False)

ratio_soum = ratio_soum.to_frame("ratio_soumission")
ratio_soum["last_name"] = ratio_soum.index.map(mapping)
print(ratio_soum.head(20))

In [ ]:
def plot_bar_top(ax,ratio, column, title, xlabel,color="#e74c3c"):
    top20_ratio = ratio.head(20).copy()

    #Label unique : nom + id
    top20_ratio["label"] = top20_ratio["last_name"] + " (" + top20_ratio.index.astype(str) + ")"

    bars = ax.barh(
        top20_ratio["label"][::-1],
        top20_ratio[column][::-1],
        color=color,
        edgecolor="white"
    )

    for bar, val in zip(bars, top20_ratio[column][::-1]):
        ax.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%",
            va="center",
            fontsize=9
        )

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, 115)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

plot_bar_top(ax,
    ratio_soum,
    "ratio_soumission",
    "Top 20 combattants les plus vulnérables aux soumissions\n(% de combats perdus par soumission)",
    "% de défaites par soumission"
)

plt.tight_layout()
plt.show()

spindola ; tes suspect mon gars mon gars

In [ ]:
# Tous les combats de Spindola
spindola = df[df['Last_Name'] == 'Spindola']
print(f"Nombre total de combats : {len(spindola)}")
print(f"\nRésultats :")
print(spindola['Résultat'].value_counts())
print(f"\nTypes de défaites :")
print(spindola[spindola['Résultat'] == 'L']['Type_victoire'].value_counts())
print(f"\nDétail des combats :")
print(spindola[['First_Name', 'Last_Name', 'Résultat', 'Type_victoire', 'Competition', 'Annee']].to_string())

on regarde ceux avec plus de 10 combats

In [ ]:
ratio_soum = (def_soum / total_combats * 100)
ratio_soum = ratio_soum[total_combats >= 10].dropna().sort_values(ascending=False)
ratio_soum = ratio_soum.to_frame("ratio_soumission")
ratio_soum["last_name"] = ratio_soum.index.map(mapping)
print(ratio_soum.head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

plot_bar_top(ax,
    ratio_soum,
    "ratio_soumission",
    "Top 20 combattants les plus vulnérables aux soumissions\n(% de combats perdus par soumission)",
    "% de défaites par soumission"
)

plt.tight_layout()
plt.show()

C'est le graphique le plus informatif. Woodmansee perd 41% de ses combats par soumission, ce qui est significativement au dessus de la moyenne du groupe (~28%).

In [ ]:
# Victoires par soumission
victoires = df[df['Résultat'] == 'W'].copy()
victoires['categorie'] = victoires['Type_victoire'].apply(categoriser)

soumissions_off = ['Étranglement', 'Lock bras/épaule', 'Lock jambe', 'Autre soumission']
victoires_soum = victoires[victoires['categorie'].isin(soumissions_off)]

# Ratio victoires par soumission / total combats
total_combats = df.groupby('id_combattant').size()
vic_soum = victoires_soum.groupby('id_combattant').size()

ratio_off = (vic_soum / total_combats * 100)
ratio_off = ratio_off[total_combats >= 10].dropna().sort_values(ascending=False)

ratio_off = ratio_off.to_frame("ratio_offense")
ratio_off["last_name"] = ratio_off.index.map(mapping)
print(ratio_off.head(20))

Griffith gagne 75% de ses combats par soumission


In [ ]:
ratio_off = ratio_off.dropna()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Offensif

plot_bar_top(axes[0],ratio_off,"ratio_offense",'Top 20 finishers\n(% victoires par soumission)',"",color='#2ecc71')


# Défensif
plot_bar_top(axes[1],ratio_soum,"ratio_soumission",'Top 20 vulnérables\n(% défaites par soumission)',"")
plt.suptitle('Profil offensif vs défensif — Soumissions', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

si t'es rouge prie pour pas qu'un vert te defense

In [ ]:
# Combiner les deux ratios
profil = pd.DataFrame({
    'offense': ratio_off["ratio_offense"],
    'defense': ratio_soum["ratio_soumission"]
}).dropna()

# Garder minimum 10 combats
profil = profil[total_combats.loc[profil.index] >= 10]

fig, ax = plt.subplots(figsize=(12, 8))

ax.scatter(profil['offense'], profil['defense'], 
           alpha=0.6, color='steelblue', s=50)

# Lignes de quadrants (moyennes)
mean_off = profil['offense'].mean()
mean_def = profil['defense'].mean()

ax.axvline(x=mean_off, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=mean_def, color='gray', linestyle='--', alpha=0.5)

# Labels quadrants
ax.text(profil['offense'].max()*0.85, profil['defense'].max()*0.9, 'Risqué', fontsize=11)
ax.text(profil['offense'].max()*0.85, profil['defense'].min()*1.1, 'Finisher', fontsize=11)
ax.text(profil['offense'].min(), profil['defense'].max()*0.9, ' Vulnérable', fontsize=11)
ax.text(profil['offense'].min(), profil['defense'].min()*1.1, ' Défensif', fontsize=11)

# Annoter les cas extrêmes
extremes = profil[(profil['offense'] > 60) | (profil['defense'] > 35)]
for name, row in extremes.iterrows():
    ax.annotate(name, (row['offense'], row['defense']),
                fontsize=7, alpha=0.8,
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('% victoires par soumission (offensive)', fontsize=12)
ax.set_ylabel('% défaites par soumission (vulnérabilité)', fontsize=12)
ax.set_title('Profil offensif vs défensif par combattant', 
             fontsize=14, fontweight='bold')


plt.tight_layout()
plt.show()

pas trop clair; illisible; mais la vraie qst :On a 518 combattants. Est-ce que Griffith à 75% d'offense c'est un vrai profil ou un accident ?"

In [ ]:
print(f"Nombre total de combattants uniques : {profil.shape[0]}")

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(12, 8))

np.random.seed(42)

for i in range(10):
    sample = profil.sample(n=100)
    ax.scatter(sample['offense'], sample['defense'],
               alpha=0.15, s=40, color='steelblue')

# Lignes de quadrants
ax.axvline(x=profil['offense'].mean(), color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=profil['defense'].mean(), color='gray', linestyle='--', alpha=0.5)

ax.text(65, 38, 'Risqué', fontsize=11)
ax.text(65, 1, 'Finisher', fontsize=11)
ax.text(2, 38, 'Vulnérable', fontsize=11)
ax.text(2, 1, 'Défensif', fontsize=11)

ax.set_xlabel('% victoires par soumission (offensive)', fontsize=12)
ax.set_ylabel('% défaites par soumission (vulnérabilité)', fontsize=12)
ax.set_title('Profil offensif vs défensif — 10 échantillons de 100 combattants', 
             fontsize=13, fontweight='bold')


plt.tight_layout()
plt.show()

Face à l'illisibilité du scatter plot complet (518 points), nous avons utilisé le bootstrap pour identifier les profils statistiquement robustes. Les combattants apparaissant systématiquement dans les zones extrêmes sont considérés comme des profils significatifs.

Afin de vérifier la robustesse des profils identifiés, nous avons appliqué une procédure de bootstrap (10 échantillons de 100 combattants sur 518). Les combattants aux profils extrêmes (Griffith, Woodmansee, Andrei...) apparaissent systématiquement dans tous les tirages, confirmant que ces profils ne sont pas des artefacts statistiques mais des caractéristiques stables.

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(12, 8))
sns.kdeplot(data=profil, x='offense', y='defense', 
            fill=True, cmap='Blues', ax=ax)
ax.axvline(x=profil['offense'].mean(), color='gray', linestyle='--')
ax.axhline(y=profil['defense'].mean(), color='gray', linestyle='--')

La zone bleu foncé correspond à la zone où se concentre la majorité des combattants. Le profil le plus courant se situe autour de 20–30 % de victoires par soumission et 5–10 % de défaites par soumission.

Le quadrant bas-gauche regroupe des combattants peu orientés vers la soumission, aussi bien en attaque qu’en défense. Cela suggère des profils plus contrôlés, potentiellement davantage orientés vers les victoires aux points.

Le quadrant bas-droite correspond aux finishers solides : ils gagnent souvent par soumission tout en se faisant rarement soumettre. À l’inverse, le quadrant haut-gauche regroupe des profils vulnérables, qui perdent souvent par soumission sans compenser par un fort taux de victoires par soumission.

Enfin, le quadrant haut-droite est peu peuplé, ce qui indique qu’il existe peu de combattants à la fois très offensifs en soumission et fortement vulnérables défensivement.

In [ ]:
q1 = profil[(profil['offense'] > mean_off) & (profil['defense'] > mean_def)]
q2 = profil[(profil['offense'] < mean_off) & (profil['defense'] > mean_def)]
q3 = profil[(profil['offense'] > mean_off) & (profil['defense'] < mean_def)]
q4 = profil[(profil['offense'] < mean_off) & (profil['defense'] < mean_def)]
print(f"Risqué: {len(q1)}, Vulnérable: {len(q2)}, Finisher: {len(q3)}, Défensif: {len(q4)}")

Analyse par compétition

In [ ]:
# Ratio soumissions par compétition
total_par_compet = df.groupby('Competition').size()
soum_par_compet = defaites_soum.groupby('Competition').size()

ratio_compet = (soum_par_compet / total_par_compet * 100)
ratio_compet = ratio_compet[total_par_compet >= 20].dropna().sort_values(ascending=False)

print(ratio_compet.head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(ratio_compet.head(20).index[::-1], 
               ratio_compet.head(20).values[::-1],
               color='#9b59b6', edgecolor='white')

for bar, val in zip(bars, ratio_compet.head(20).values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.axvline(x=ratio_compet.head(20).values.mean(), 
           color='navy', linestyle='--', alpha=0.7,
           label=f'Moyenne : {ratio_compet.head(20).values.mean():.1f}%')

ax.set_title('Compétitions avec le plus de soumissions\n(min. 20 combats)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('% de combats terminés par soumission')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 40)

plt.tight_layout()
plt.show()

Le graphique révèle une distinction claire entre deux types de compétitions :

Au dessus de la moyenne (24.2%)
Quintet 3, Queen of Mats, Emerald City, ACB JJ. Ce sont des compétitions invitation only ou à format spécial (no-time limit, soumission only). Le règlement pousse naturellement les combattants à chercher la soumission plutôt que de gérer le temps.

En dessous de la moyenne
IBJJF Crown, WNO, CJI : ce sont des compétitions traditionnelles avec points où la gestion du score prime. Les combattants jouent la sécurité.

Observation clé
Même la compétition la plus offensive (Quintet 3 à 32%) ne dépasse pas le tiers des combats terminés par soumission, ce qui confirme ce qu'on avait vu dans la distribution globale : le BJJ compétitif moderne est dominé par les points, même dans les formats les plus offensifs.

In [ ]:
# Ratio soumissions par année
total_par_annee = df.groupby('Annee').size()
soum_par_annee = defaites_soum.groupby('Annee').size()

ratio_annee = (soum_par_annee / total_par_annee * 100).dropna()

print(ratio_annee)

In [ ]:
# Garder uniquement les années avec assez de données
ratio_annee = ratio_annee[total_par_annee >= 50]
ratio_annee = ratio_annee[ratio_annee.index >= 2010]

print(ratio_annee)

2026 on a que quelques mois de data, on le vire

In [ ]:
ratio_annee = ratio_annee[ratio_annee.index < 2026]

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(ratio_annee.index, ratio_annee.values, 
        color='#e74c3c', linewidth=2.5, marker='o', markersize=6)

# Tendance
z = np.polyfit(ratio_annee.index, ratio_annee.values, 1)
p = np.poly1d(z)
ax.plot(ratio_annee.index, p(ratio_annee.index), 
        color='navy', linestyle='--', alpha=0.7, label=f'Tendance')

# Remplir sous la courbe
ax.fill_between(ratio_annee.index, ratio_annee.values, 
                alpha=0.1, color='#e74c3c')

ax.set_title('Évolution du taux de soumissions en BJJ (2010-2025)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Année')
ax.set_ylabel('% de combats terminés par soumission')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks(ratio_annee.index)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

La tendance générale est à la hausse légère mais constante des soumissions sur 15 ans, passant de ~5% en 2010 à ~9% en 2025.

Points notables :

2014 : pic à 9% puis chute brutale en 2015. Probablement lié à un changement de règlement IBJJF en 2015 qui a restreint certaines techniques (heel hooks notamment en Gi), poussant les combattants vers les points.

2018 : creux à ~7%. Coïncide avec une période de domination des compétitions traditionnelles à points dans le dataset.

2019-2025 : remontée stable et progressive. Correspond à l'essor du NoGi et des compétitions alternatives (WNO, ADCC, Quintet) qui autorisent plus de techniques de jambes et favorisent les soumissions.

Conclusion principale : Le BJJ compétitif devient progressivement plus offensif, tiré par le développement du NoGi et des formats alternatifs. Cependant les niveaux restent bas (5-9%) confirmant la domination des points sur l'ensemble de la période.

In [ ]:
# Features par combattant
features = pd.DataFrame({
    'ratio_offense': ratio_off['ratio_offense'],
    'ratio_defense': ratio_soum['ratio_soumission'],
    'total_combats': total_combats,
    'ratio_victoires': df.groupby('id_combattant')['Résultat'].apply(
        lambda x: (x == 'W').sum() / len(x) * 100
    )
}).dropna()

# Garder min 10 combats
features = features[features["total_combats"] >= 10]

print(features.shape)
print(features.head())

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Normalisation
scaler = StandardScaler()
X = scaler.fit_transform(features)
profil = pd.DataFrame({
    'offense': ratio_off["ratio_offense"],
    'defense': ratio_soum["ratio_soumission"]
}).dropna()
# Choisir le bon k avec la méthode du coude
inerties = []
K = range(2, 10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inerties.append(kmeans.inertia_)

# Tracer le coude
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K, inerties, marker='o', color='steelblue', linewidth=2)
ax.set_title('Méthode du coude — Choix du K optimal', fontweight='bold')
ax.set_xlabel('Nombre de clusters K')
ax.set_ylabel('Inertie')
plt.tight_layout()
plt.show()

Le coude est pas très marqué mais on voit un changement de pente autour de K=4 — la courbe s'aplatit progressivement après. K=4 c'est logique aussi car on a naturellement 4 quadrants dans notre scatter plot (Finisher, Défensif, Vulnérable, Risqué).

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
features['cluster'] = kmeans.fit_predict(X)

# Voir les caractéristiques moyennes de chaque cluster
print(features.groupby('cluster').mean().round(2))

Les 4 clusters sont lisibles et peuvent être interprétés à partir des moyennes de leurs variables :

Cluster 0 — "Vulnérables"  
Offense modérée/faible (23 %), vulnérabilité élevée (18 %) et taux de victoires plus faible (55 %). Ce groupe correspond aux combattants les plus fragiles défensivement, car ils subissent davantage de défaites par soumission.

Cluster 1 — "Finishers"  
Offense très élevée (47 %), vulnérabilité faible (6 %) et très bon taux de victoires (77 %). Ce groupe correspond aux combattants les plus efficaces en soumission : ils gagnent souvent par soumission tout en se faisant rarement soumettre.

Cluster 2 — "Techniciens"  
Offense modérée/faible (22 %), vulnérabilité faible (7 %) et bon taux de victoires (67 %). Ce groupe correspond à des combattants solides, moins orientés vers la soumission que les finishers, mais capables de gagner régulièrement, probablement grâce à un style plus contrôlé.

Cluster 3 — "Vétérans"  
Offense correcte (31 %), vulnérabilité faible (7 %), bon taux de victoires (74 %) et surtout un nombre moyen de combats beaucoup plus élevé (185). Ce groupe correspond aux combattants les plus expérimentés de la base.

In [ ]:
colors = {0: '#2ecc71', 1: '#3498db', 2: '#e74c3c', 3: '#f39c12'}
labels = {
    0: "Vulnérables",
    1: "Finishers",
    2: "Techniciens",
    3: "Vétérans"
}

fig, ax = plt.subplots(figsize=(12, 8))

for cluster in range(4):
    data = features[features['cluster'] == cluster]
    ax.scatter(data['ratio_offense'], data['ratio_defense'],
               color=colors[cluster], label=labels[cluster],
               alpha=0.7, s=60)

ax.axvline(x=features['ratio_offense'].mean(), color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=features['ratio_defense'].mean(), color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('% victoires par soumission (offensive)', fontsize=12)
ax.set_ylabel('% défaites par soumission (vulnérabilité)', fontsize=12)
ax.set_title('Clustering K-means des combattants BJJ (K=4)', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

Le clustering est visuellement cohérent : les 4 groupes se distinguent principalement selon leur rapport aux soumissions et leur expérience.

Les vulnérables (vert) se situent majoritairement dans la partie haute du graphique : ils ont un taux de défaites par soumission élevé. Ce groupe correspond donc aux combattants les plus fragiles défensivement.

Les finishers (bleu) se trouvent surtout à droite et plutôt en bas : ils gagnent souvent par soumission tout en se faisant rarement soumettre. C’est le groupe le plus offensif et le plus solide vis-à-vis des soumissions.

Les techniciens (rouge) sont plutôt regroupés dans la partie basse/gauche ou centrale : ils ont une offense par soumission plus faible que les finishers, mais restent relativement peu vulnérables. Cela suggère un style plus contrôlé, moins centré sur la finalisation par soumission.

Les vétérans (orange) sont plus difficiles à distinguer uniquement sur ce scatter plot, car leur caractéristique principale n’est pas visible ici : ils se démarquent surtout par un nombre de combats beaucoup plus élevé. Ce graphique ne montre donc que 2 dimensions parmi les 4 utilisées dans le clustering.

In [ ]:
import seaborn as sns

# Matrice de corrélation
corr = features[['ratio_offense', 'ratio_defense', 'total_combats', 'ratio_victoires']].corr()
print(corr.round(2))

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, ax=ax, square=True)
ax.set_title('Matrice de corrélation des features', fontweight='bold')
plt.tight_layout()
plt.show()

ratio_offense ↔ ratio_victoires : +0.52 - logique, plus tu soumets, plus tu gagnes.


ratio_defense ↔ ratio_victoires : -0.61 - plus tu te fais soumettre, moins tu gagnes. C'est la corrélation la plus forte.

ratio_offense ↔ ratio_defense : -0.24 - légère corrélation négative, les bons offensifs se font moins soumettre.

total_combats ↔ ratio_defense : -0.17 - les vétérans (beaucoup de combats) se font moins soumettre, l'expérience protège.

les 4 variables sont bien corrélées entre elles, particulièrement autour du ratio_victoires qui est lié aux 3 autres. La PCA est donc justifiée, elle va condenser ces redondances en 2 axes synthétiques.

on fait une pca

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"Variance expliquée : PC1={pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"Variance expliquée : PC2={pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Total : {sum(pca.explained_variance_ratio_)*100:.1f}%")

# Contribution des variables
print("\nContribution des variables :")
for i, var in enumerate(['ratio_offense', 'ratio_defense', 'total_combats', 'ratio_victoires']):
    print(f"PC1: {var} = {pca.components_[0][i]:.2f} | PC2: {var} = {pca.components_[1][i]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

colors = {0: '#2ecc71', 1: '#3498db', 2: '#e74c3c', 3: '#f39c12'}
labels = {
    0: "Vulnérables",
    1: "Finishers",
    2: "Techniciens",
    3: "Vétérans"
}

for cluster in range(4):
    mask = features['cluster'] == cluster
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=colors[cluster], label=labels[cluster],
               alpha=0.7, s=60)

plt.xlabel("PC1 : Profil de performance / style(50.4%)", fontsize = 12)
ax.set_ylabel('PC2 : Expérience (24.7%)', fontsize=12)
ax.set_title('PCA des combattants BJJ — 74.6% de variance expliquée', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=["ratio_offense", "ratio_defense", "total_combats", "ratio_victoires"],
    columns=["PC1", "PC2"]
)

print(pca_loadings.round(2))

PC1 (axe horizontal : opposition entre profils de performance)

La séparation de gauche à droite ne doit pas être interprétée comme un simple axe de niveau ou de qualité. Elle oppose plutôt différents profils de combattants.

Les vulnérables (vert) sont majoritairement situés à gauche : ils se distinguent surtout par une vulnérabilité plus élevée aux soumissions.

Les finishers (bleu) sont davantage situés à droite et en bas : ils correspondent aux combattants qui gagnent souvent par soumission tout en subissant peu de défaites par soumission.

Les techniciens (rouge) occupent une position plus centrale : ils ont un profil plus équilibré, moins orienté vers la soumission que les finishers, mais avec une bonne solidité défensive.

PC2 (axe vertical : expérience)

Le deuxième axe semble surtout isoler les vétérans (orange), qui sont clairement situés plus haut. Cette séparation est cohérente avec le fait que ce cluster est principalement caractérisé par un nombre de combats beaucoup plus élevé.

Les autres clusters restent davantage regroupés autour de 0 sur cet axe, ce qui indique que leur différence principale ne vient pas de l’expérience, mais plutôt de leur style de combat.

La PCA confirme et améliore le clustering — les 4 profils sont bien distincts dans l'espace réduit. Le résultat le plus intéressant : l'expérience et le niveau sont deux dimensions indépendantes — un vétéran n'est pas forcément un finisher.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_names = ['ratio_offense', 'ratio_defense', 'total_combats', 'ratio_victoires']
titles = ['% Victoires par soumission (offense)', 
          '% Défaites par soumission (defense)',
          'Nombre total de combats',
          '% Victoires global']
colors_list = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
labels = ['Vulnérables', 'Finishers', 'Techniciens', 'Vétérans']

for ax, feature, title in zip(axes.flatten(), features_names, titles):
    data = [features[features['cluster'] == k][feature].values for k in range(4)]
    bp = ax.boxplot(data, patch_artist=True, labels=labels)
    
    for patch, color in zip(bp['boxes'], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Distribution des features par cluster', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

% Victoires par soumission (offense)

Les finishers sont clairement au-dessus des autres groupes, avec une médiane autour de 45 %. C’est leur caractéristique principale : ils remportent une grande partie de leurs combats par soumission.

Les vulnérables et les techniciens ont des niveaux d’offense plus proches, autour de 20–25 %. Cette variable seule ne suffit donc pas à les distinguer nettement.

Les vétérans occupent une position intermédiaire, avec une offense correcte mais moins marquée que celle des finishers.

% Défaites par soumission (defense)

Les vulnérables se distinguent très nettement, avec une médiane autour de 17 % et plusieurs valeurs extrêmes au-dessus de 30 %. C’est leur caractéristique principale : ils subissent plus souvent des défaites par soumission.

Les finishers, techniciens et vétérans restent beaucoup plus bas, généralement autour de 5–8 %, ce qui montre une meilleure solidité défensive face aux soumissions.

Nombre total de combats

Les vétérans sont clairement séparés des autres groupes : leur nombre total de combats est beaucoup plus élevé, avec une médiane autour de 170–180 combats et plusieurs valeurs extrêmes au-delà de 300.

Les trois autres groupes ont des volumes de combats plus faibles et relativement proches, même si quelques outliers existent.

% Victoires global

Les finishers et les vétérans présentent les meilleurs taux de victoire globaux, autour de 75 %. Cela indique que ces deux profils sont globalement les plus performants.

Les techniciens ont un taux de victoire correct, autour de 65–70 %, ce qui confirme un profil solide mais moins dominant.

Les vulnérables sont clairement en dessous, avec une médiane autour de 55 % et une dispersion importante. Cela confirme que leur forte vulnérabilité aux soumissions est associée à une performance globale plus faible.

clus spectral 

Le clustering spectral appliqué sur les mêmes features que le K-means produit un résultat dégénéré — un cluster domine largement les autres. Ceci s'explique par le fait que le clustering spectral est optimisé pour des données avec des structures géométriques complexes (formes non convexes), alors que nos données tabulaires ont une structure globulaire mieux adaptée au K-means.

In [ ]:
from sklearn.neighbors import kneighbors_graph
import networkx as nx

# Construire le graphe des k plus proches voisins
A = kneighbors_graph(X, n_neighbors=5, mode='connectivity', include_self=False)

# Convertir en graphe networkx
G = nx.from_scipy_sparse_array(A)

# Ajouter les labels de cluster comme attribut
for i, (name, row) in enumerate(features.iterrows()):
    G.nodes[i]['cluster'] = int(row['cluster'])
    G.nodes[i]['name'] = name

print(f"Noeuds : {G.number_of_nodes()}")
print(f"Arêtes : {G.number_of_edges()}")

Concrètement la matrice A c'est une matrice 518×518 où A[i][j] = 1 si le combattant j est parmi les 5 plus proches voisins de i, sinon 0.

Le graphe est construit avec l'algorithme des K plus proches voisins (KNN) :
Pour chaque combattant on calcule sa distance avec tous les autres dans l'espace des 4 features normalisées, et on garde uniquement les 5 plus proches — on trace une arête entre eux.

1. Calculer la distance euclidienne avec les 517 autres
   distance(Abreu, Agrizzi) = √((offense_A - offense_B)² + (defense_A - defense_B)² + ...)

2. Garder les 5 plus proches
   Abreu → [Agrizzi, Agazarm, Costa, Lima, Santos]

3. Tracer une arête entre eux
   Abreu ── Agrizzi
   Abreu ── Agazarm
   ...

In [ ]:
# Degré de chaque noeud
degree = dict(G.degree())
top_hubs = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 combattants les plus connectés :")
for node, deg in top_hubs:
    name = G.nodes[node]['name']
    cluster = G.nodes[node]['cluster']
    print(f"{name} — degré {deg} — {labels_map[cluster]}")

In [ ]:
print(features['cluster'].value_counts())

In [ ]:
# Vérifier les labels dans le graphe
clusters_in_graph = [G.nodes[i]['cluster'] for i in G.nodes()]
print(pd.Series(clusters_in_graph).value_counts())



"Le graphe de similarité KNN (K=5) montre que les combattants forment un réseau dense sans communautés clairement séparées visuellement, ce qui confirme que les frontières entre clusters sont graduelles plutôt que nettes — cohérent avec les résultats du K-means où Techniciens et Finishers se chevauchent partiellement."

In [ ]:
import choix
import numpy as np

# Index numérique pour chaque combattant
combattants = df['Combattant'].unique()
idx = {name: i for i, name in enumerate(combattants)}

print(f"Nombre de combattants : {len(idx)}")

# Construire les paires (gagnant, perdant)
paires = []
for _, row in df.iterrows():
    if row['Résultat'] == 'W':
        gagnant = row['Combattant']
        perdant = row['Adversaire']
        
        # Vérifier que les deux sont dans l'index
        if gagnant in idx and perdant in idx:
            paires.append((idx[gagnant], idx[perdant]))

print(f"Nombre de paires construites : {len(paires)}")
print(f"Exemple : {paires[:5]}")

In [ ]:
# Construire le graphe des affrontements
G_bt = nx.DiGraph()
for gagnant, perdant in paires:
    G_bt.add_edge(gagnant, perdant)

# Garder uniquement la composante connexe principale
composantes = list(nx.weakly_connected_components(G_bt))
print(f"Nombre de composantes : {len(composantes)}")
print(f"Taille de la plus grande : {max(len(c) for c in composantes)}")

# Garder la plus grande
principale = max(composantes, key=len)
print(f"Combattants dans la composante principale : {len(principale)}")

# Filtrer les paires
paires_filtrees = [(g, p) for g, p in paires if g in principale and p in principale]

# Réindexer
idx_inverse = {v: k for k, v in idx.items()}
combattants_principaux = [idx_inverse[i] for i in principale]
idx_new = {name: i for i, name in enumerate(combattants_principaux)}

paires_reindexees = [(idx_new[idx_inverse[g]], idx_new[idx_inverse[p]]) 
                     for g, p in paires_filtrees]

print(f"Paires filtrées : {len(paires_reindexees)}")

In [ ]:
# Vérifier combien de combats ont les deux combattants dans le dataset
df_both = df[df['Adversaire'].isin(df['Combattant'].unique())]
print(f"Combats où l'adversaire est aussi dans le dataset : {len(df_both)}")
print(f"Sur total : {len(df)}")
print(f"Soit : {len(df_both)/len(df)*100:.1f}%")

bT n'est pas appliquable ici


"Le modèle de Bradley-Terry aurait permis d'estimer un score de force relatif pour chaque combattant à partir des affrontements directs. Cependant, seulement 0.1% des combats du dataset ont leurs deux participants référencés, rendant le graphe d'affrontements trop sparse (38 combattants connectés sur 518) pour que le modèle converge. Cette limite structurelle du dataset empêche l'application de modèles comparatifs et suggère qu'un dataset plus complet (type base de données officielle IBJJF) serait nécessaire."

suite eda

In [ ]:
# 1 - Heatmap type de soumission x catégorie de poids
soumissions_victoires = victoires[victoires['categorie'].isin(soumissions_off)].copy()

# Nettoyer les poids
soumissions_victoires = soumissions_victoires[soumissions_victoires['Weight'].notna()]

# Garder les catégories de poids les plus fréquentes
top_weights = soumissions_victoires['Weight'].value_counts().head(8).index
soumissions_victoires = soumissions_victoires[soumissions_victoires['Weight'].isin(top_weights)]

# Construire la heatmap
heatmap_data = pd.crosstab(soumissions_victoires['categorie'], 
                            soumissions_victoires['Weight'],
                            normalize='columns') * 100

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=ax, cbar_kws={'label': '% des soumissions'})
ax.set_title('Type de soumission par catégorie de poids (%)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Catégorie de poids')
ax.set_ylabel('Type de soumission')
plt.tight_layout()
plt.show()

In [ ]:
# 2 - Soumissions les plus létales
total_combats_global = len(df)
soum_par_type = victoires[victoires['categorie'].isin(soumissions_off)]['categorie'].value_counts()
letalite = (soum_par_type / total_combats_global * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(letalite.index[::-1], letalite.values[::-1], color='#e74c3c', edgecolor='white')
for bar, val in zip(bars, letalite.values[::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=9)
ax.set_title('Létalité des types de soumission\n(% de tous les combats terminés par ce type)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('% de combats terminés')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# 3 - Evolution des soumissions dans le temps
soum_temps = victoires[victoires['categorie'].isin(soumissions_off)].copy()
soum_temps = soum_temps[soum_temps['Annee'] >= 2010]
soum_temps = soum_temps[soum_temps['Annee'] < 2026]

evol = soum_temps.groupby(['Annee', 'categorie']).size().unstack(fill_value=0)
evol = evol.div(evol.sum(axis=1), axis=0) * 100

colors_evol = {'Étranglement': '#3498db', 'Lock bras/épaule': '#2ecc71',
               'Lock jambe': '#e74c3c', 'Autre soumission': '#f39c12'}

fig, ax = plt.subplots(figsize=(12, 6))
for col in evol.columns:
    ax.plot(evol.index, evol[col], marker='o', linewidth=2,
            label=col, color=colors_evol.get(col, 'gray'))

ax.set_title('Évolution des types de soumission (2010-2025)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Année')
ax.set_ylabel('% des soumissions')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks(evol.index)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# 4 - Profil de soumission des finishers
finishers = features[features['cluster'] == 0].index.tolist()

profil_finishers = victoires[
    (victoires['id_combattant'].isin(finishers)) & 
    (victoires['categorie'].isin(soumissions_off))
].groupby(['id_combattant', 'categorie']).size().unstack(fill_value=0)

# Normaliser par combattant
profil_finishers = profil_finishers.div(profil_finishers.sum(axis=1), axis=0) * 100

# Garder top 15 finishers par total de soumissions
top_finishers = victoires[
    (victoires['id_combattant'].isin(finishers)) & 
    (victoires['categorie'].isin(soumissions_off))
].groupby('id_combattant').size().sort_values(ascending=False).head(15).index

profil_finishers = profil_finishers.loc[profil_finishers.index.isin(top_finishers)]

fig, ax = plt.subplots(figsize=(12, 8))
profil_finishers.plot(kind='barh', stacked=True, ax=ax,
                      color=[colors_evol.get(c, 'gray') for c in profil_finishers.columns])
ax.set_title('Profil de soumission des top 15 Finishers', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('% des victoires par soumission')
ax.legend(fontsize=10, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Analyse des matches remporté par points

On a étudié les victoires / défaites par soumissions. On va maintenant essayer de comparer ça aux victoires/défaites par points.

Tout d'abord, j'aimerais préciser que cette partie ne pourra pas être aussi étudié que la précédente car il nous manque une information importante qu'on n'a pas pu obtenir. Comment on été gagner ces points( passage de garde, prise de posisition forte...) et quand est ce qu'ils ont été gagné.

In [ ]:
# Filtrer uniquement les points
defaites_points  = defaites_clean[defaites_clean['categorie']=='Points/Avantages']
par_combattant   = defaites_points.groupby('id_combattant').size().sort_values(ascending=False)

In [ ]:
def_points = defaites_points.groupby('id_combattant').size()
ratio_point = (def_points / total_combats * 100).dropna().sort_values(ascending=False)

print(ratio_point.head(20))


In [ ]:
print("Statistique ratio point")
print(ratio_point.describe())
print("Statistique ration soumission")
print(ratio_soum.describe())

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.histplot(ratio_point)
plt.title("Distribution ratio point")

plt.subplot(1,2,2)
sns.histplot(ratio_soum)
plt.title("Distribution ratio soumission")

plt.tight_layout()

La distribution de ratio_point présente une forme relativement symétrique en cloche, suggérant une proximité avec une distribution normale.
En revanche, la distribution de ratio_soum est fortement asymétrique à droite, avec une concentration de valeurs faibles et une longue queue vers les grandes valeurs.

En comparant cela avec notre describe. On conclu que les personnes se faisant soumettre, se font moins soumettre que les personnes perdants par points. 

Cela explique le nombre plus important de défaites/victoires par points que par soumissions.

Le fait que la distribution de ratio_point soit proche d’une loi normale suggère que cette variable résulte de la combinaison de nombreux facteurs aléatoires indépendants.

In [ ]:
# Ratio points par année
ratio_annee_soum = ratio_annee

point_par_annee = defaites_points.groupby('Annee').size()

ratio_annee_point = (point_par_annee / total_par_annee * 100).dropna()
ratio_annee_point = ratio_annee_point[total_par_annee >= 50]
ratio_annee_point = ratio_annee_point[ratio_annee_point.index >= 2010]
ratio_annee_point = ratio_annee_point[ratio_annee_point.index < 2026]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(ratio_annee_point.index, ratio_annee_point.values, 
        color='#e74c3c', linewidth=2.5, marker='o', markersize=6,
        label='Ratio point')
ax.plot(ratio_annee_soum.index, ratio_annee_soum.values, 
        color='#3498db', linewidth=2.5, marker='o', markersize=6,
        label='Ratio soumission')


z = np.polyfit(ratio_annee_point.index, ratio_annee_point.values, 1)
p = np.poly1d(z)
ax.plot(ratio_annee_point.index, p(ratio_annee_point.index), 
        color='#e74c3c', linestyle='--', alpha=0.7, label=f'Tendance point')

z = np.polyfit(ratio_annee_soum.index, ratio_annee_soum.values, 1)
p = np.poly1d(z)
ax.plot(ratio_annee_soum.index, p(ratio_annee_soum.index), 
        color='#3498db', linestyle='--', alpha=0.7, label=f'Tendance soumission')

# Remplissage
ax.fill_between(ratio_annee_point.index, ratio_annee_point.values, 
                alpha=0.1, color='#e74c3c')
ax.fill_between(ratio_annee_soum.index, ratio_annee_soum.values, 
                alpha=0.1, color='#3498db')

ax.set_title('Évolution du taux de soumissions/point en BJJ (2010-2025)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Année')
ax.set_ylabel('% de combats terminés par soumission')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xticks(ratio_annee_point.index)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

On remarque malgré tout que ces dernières années le ratio de soumissions augmente tandis que le ratio de victoire par points diminue.

Cela est notament du au fait que les fédérations de jjb souhaite de plus en plus pousser les personnes à soumettre leurs adversaire.

In [ ]:
corr_features = features.copy()
corr_features = corr_features.drop(columns={'cluster','cluster_spectral'})

ratio_point.name = "ratio_point"

corr_features = corr_features.join(ratio_point, how="inner")

In [ ]:
corr_features

In [ ]:
corr_features.corr()['ratio_point']

Peu intéressant car, les variables sont construits à partir des mêmes informations.

In [ ]:
corr_df = df.copy()
corr_df['categorie'] = corr_df['Type_victoire'].apply(categoriser)
corr_df = corr_df.drop(columns ={'URL','Stage','first_name','last_name','Type_victoire','Adversaire','Competition','Combattant','First_Name','Last_Name'})
corr_df["ratio_point"] = corr_df["id_combattant"].map(ratio_point)

In [ ]:
corr_df['Weight'] = corr_df['Weight'].str.replace(r'(\d+)KG', r'\1', regex=True)
corr_df['Weight'] = pd.to_numeric(corr_df['Weight'], errors='coerce')
corr_df['Weight'] = corr_df['Weight'].fillna(
    corr_df.groupby('id_combattant')['Weight'].transform('mean')
)

corr_df['Résultat'] = corr_df['Résultat'] == 'W'
corr_df_ohe = pd.get_dummies(corr_df, columns=["categorie"]) # One hot encoding

In [ ]:
corr_df_ohe.corr()['ratio_point']

Les corrélations observées sont faibles, ce qui ne signifie pas une absence de relation, mais plutôt une inadéquation de la métrique utilisée. En effet, le ratio_point est une variable agrégée au niveau du combattant, tandis que les variables explicatives sont au niveau des combats individuels.

Une analyse de chacune des variables est donc plus intéressantes.

In [ ]:
sns.boxplot(data=corr_df, x="categorie", y="ratio_point")
plt.xticks(rotation=45)

Le boxplot ne met pas en évidence de différences marquées entre les catégories de victoire en termes de ratio_point. Les distributions sont très similaires, avec des médianes proches et une forte dispersion. Cela suggère que le type de victoire n’influence pas significativement la proportion de victoires aux points d’un combattant.

Essayons d'analyser l'impact du poids sur les points.

In [ ]:
ratio_par_poids = corr_df.groupby("Weight")["ratio_point"].mean()

plt.figure(figsize=(10,6))
ratio_par_poids.sort_index().plot(marker='o')

plt.xlabel("Poids")
plt.ylabel("Ratio point moyen")
plt.title("Ratio point moyen en fonction du poids")

plt.grid(alpha=0.3)
plt.show()

Ce plot est illisible et ininterprétable. 
De plus au Jujitsu Bresilien il existe des categories de poids.
Une analyse par catégories est donc plus pertinentes.

In [ ]:
bins = [0, 57.5, 64, 70, 76, 82.3, 88.3, 94.3, 100.5, 200]
labels = [
    "Galo", "Pluma", "Pena", "Leve",
    "Medio", "Meio Pesado", "Pesado",
    "Super Pesado", "Pesadissimo"
]

corr_df["poids_cat"] = pd.cut(corr_df["Weight"], bins=bins, labels=labels)

In [ ]:
ratio_par_cat = corr_df.groupby("poids_cat")["ratio_point"].mean()

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.bar(ratio_par_cat.index,ratio_par_cat.values)
plt.xlabel("Catégorie de poids (IBJJF)")
plt.ylabel("Ratio point moyen")
plt.title("Ratio point moyen par catégorie de poids")

plt.grid(axis="y", alpha=0.3)
plt.show()

On en conclu que la catégories de poids n'a pas d'importance sur le ratio de points de chaque combattant.

In [ ]:
plt.figure(figsize=(8,6))

scatter = plt.scatter(
    corr_features["ratio_offense"],
    corr_features["ratio_defense"],
    c=corr_features["ratio_point"],
    cmap="coolwarm",
    alpha=0.7
)

plt.colorbar(scatter, label="% victoires aux points")

plt.xlabel("% victoires par soumission (offense)")
plt.ylabel("% défaites par soumission (défense)")
plt.title("Profils de combattants + stratégie points")

plt.show()

On observe une opposition entre deux styles de combattants : les finishers, qui privilégient les soumissions et gagnent rarement aux points, et des profils plus stratégiques qui remportent davantage leurs combats aux points. 

En croisant cette observation avec le clustering réalisé précédemment, on retrouve clairement le profil des techniciens. Ce sont des combattants moins orientés vers la soumission, mais plus efficaces dans les victoires aux points.

In [ ]:
corr_df

In [ ]:
victoires_point = corr_df[corr_df["Résultat"] & (corr_df["categorie"]=="Points/Avantages")]
defaites_point = corr_df[~corr_df["Résultat"] & (corr_df["categorie"]=="Points/Avantages")]

total_combats = corr_df.groupby("id_combattant").size()

ratio_off_point = victoires_point.groupby("id_combattant").size() / total_combats * 100
ratio_def_point = defaites_point.groupby("id_combattant").size() / total_combats * 100

total_points = (
    corr_df[corr_df["categorie"]=="Points/Avantages"]
    .groupby("id_combattant")
    .size()
)

In [ ]:
features_points = pd.DataFrame({
    "ratio_off_point": ratio_off_point,
    "ratio_def_point": ratio_def_point,
    "total_points": total_points,
    "total_combats": total_combats
}).fillna(0)

features_points = features_points[features_points["total_combats"] >= 10]

In [ ]:
features_points

On va maintenant chercher à comparer les différentes catégories de personnes gagnants et perdants par points à nos personnes gagnant et perdant par soumissions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    features_points["ratio_off_point"],
    features_points["ratio_def_point"],
    alpha=0.6
)

ax.axvline(features_points["ratio_off_point"].mean(), linestyle="--", color="gray", alpha=0.5)
ax.axhline(features_points["ratio_def_point"].mean(), linestyle="--", color="gray", alpha=0.5)

ax.set_xlabel("% victoires aux points")
ax.set_ylabel("% défaites aux points")
ax.set_title("Profil offensif vs défensif — Victoires/Défaites aux points")

plt.tight_layout()
plt.show()

In [ ]:
X_points = features_points[[
    "ratio_off_point",
    "ratio_def_point",
    "total_points",
    "total_combats"
]]

scaler_point = StandardScaler()
X_points_scaled = scaler_point.fit_transform(X_points)


In [ ]:
# Choisir le bon k avec la méthode du coude
inerties = []
K = range(2, 10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_points_scaled)
    inerties.append(kmeans.inertia_)

# Tracer le coude
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K, inerties, marker='o', color='steelblue', linewidth=2)
ax.set_title('Méthode du coude — Choix du K optimal', fontweight='bold')
ax.set_xlabel('Nombre de clusters K')
ax.set_ylabel('Inertie')
plt.tight_layout()
plt.show()

In [ ]:
kmeans_points = KMeans(n_clusters=4, random_state=42, n_init=10)
features_points["cluster_points"] = kmeans_points.fit_predict(X_points_scaled)

In [ ]:
features_points.groupby("cluster_points")[[
    "ratio_off_point",
    "ratio_def_point",
    "total_points",
    "total_combats"
]].mean().round(2)

Les 4 clusters sont lisibles et peuvent être interprétés à partir des moyennes de leurs variables :

Cluster 0 : "Peu orientés points"  
Ratio de victoires aux points faible (17 %), ratio de défaites aux points faible (11 %) et faible nombre moyen de combats aux points (20). Ce groupe correspond aux combattants dont les combats se terminent moins souvent aux points, ce qui suggère un style davantage orienté vers d’autres issues, comme les soumissions.

Cluster 1 : "Vulnérables aux points"  
Ratio de victoires aux points moyen (27 %), mais ratio de défaites aux points élevé (25 %). Ce groupe correspond aux combattants les plus fragiles lorsque le combat se joue aux points : ils perdent souvent dans ce type de situation sans compenser par un taux de victoires suffisamment élevé.

Cluster 2 : "Vétérans"  
Ratio de victoires aux points élevé (34 %), ratio de défaites aux points modéré (15 %) et surtout un nombre moyen de combats aux points très élevé (96) et de total de combats aussi(180). Ce groupe correspond aux combattants dont les combats vont fréquemment à la décision, avec une capacité importante à gagner dans ce cadre.

Cluster 3 : "Efficaces aux points"  
Meilleur ratio de victoires aux points (42 %) et ratio de défaites aux points faible (12 %). Ce groupe ne correspond pas forcément aux combattants qui vont le plus souvent aux points, mais plutôt à ceux qui sont les plus efficaces lorsque le combat se termine aux points.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

labels = {
    0: "Peu orientés points",
    1: "Vulnérables aux points",
    2: "Vétérans",
    3: "Efficace aux points"
}

colors = {
    0: "#e74c3c",
    1: "#2ecc71",
    2: "#f39c12",
    3: "#3498db" 
}

for cluster in range(4):
    data = features_points[features_points["cluster_points"] == cluster]
    
    ax.scatter(
        data["ratio_off_point"],
        data["ratio_def_point"],
        color=colors[cluster],
        label=labels[cluster],
        alpha=0.7,
        s=60
    )

ax.axvline(
    x=features_points["ratio_off_point"].mean(),
    color="gray",
    linestyle="--",
    alpha=0.5
)

ax.axhline(
    y=features_points["ratio_def_point"].mean(),
    color="gray",
    linestyle="--",
    alpha=0.5
)

ax.set_xlabel("% victoires aux points", fontsize=12)
ax.set_ylabel("% défaites aux points", fontsize=12)

ax.set_title(
    "Clustering des profils aux points",
    fontsize=13,
    fontweight="bold"
)

ax.legend(fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

Le clustering aux points est visuellement cohérent : les 4 groupes se distinguent principalement selon leur rapport aux victoires et défaites aux points.

Les vulnérables aux points (vert) se situent majoritairement dans la partie haute du graphique : ils ont un taux de défaites aux points élevé. Ce groupe correspond donc aux combattants les plus fragiles lorsque le combat se décide aux points.

Les vétérans (orange) se trouvent plutôt dans la zone droite/centrale : ils ont un taux de victoires aux points élevé et un volume important de combats terminés aux points. Ce sont des combattants dont le style semble souvent mener à des décisions aux points.

Les efficaces aux points (rouge) se situent surtout à droite et plutôt en bas : ils gagnent souvent aux points tout en perdant relativement peu aux points. Ce groupe correspond donc aux combattants les plus performants lorsque le combat se joue aux points.

Les peu orientés points (bleu) se trouvent surtout à gauche et en bas : ils gagnent peu aux points, mais perdent aussi peu aux points. Cela suggère que leurs combats se terminent moins souvent aux points et davantage par d’autres issues, comme les soumissions.

In [ ]:
features_points_box = features_points.copy()

# Ajouter le ratio de victoires global
features_points_box["ratio_victoires"] = corr_features["ratio_victoires"].reindex(features_points_box.index)

# Supprimer les lignes sans ratio_victoires
features_points_box = features_points_box.dropna(subset=["ratio_victoires"])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_names = [
    "ratio_off_point",
    "ratio_def_point",
    "total_points",
    "ratio_victoires"
]

titles = [
    "% Victoires aux points",
    "% Défaites aux points",
    "Nombre total de combats aux points",
    "% Victoires global"
]

colors_list = ["#e74c3c", "#2ecc71","#f39c12" , "#3498db"]

labels = [
    "Peu orientés\npoints",
    "Vulnérables\naux points",
    "Vétérans",
    "Efficaces\naux points"
    
]

for ax, feature, title in zip(axes.flatten(), features_names, titles):
    data = [
        features_points_box[features_points_box["cluster_points"] == k][feature].dropna().values
        for k in range(4)
    ]

    bp = ax.boxplot(data, patch_artist=True, tick_labels=labels)

    for patch, color in zip(bp["boxes"], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(title, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", rotation=15)

plt.suptitle(
    "Distribution des features points par cluster",
    fontsize=14,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
plt.show()

Les boxplots confirment que les clusters aux points correspondent bien à des profils différents.

% Victoires aux points

Les efficaces aux points ont le ratio de victoires aux points le plus élevé. C’est leur caractéristique principale : lorsqu’un combat se termine aux points, ils ont tendance à le gagner.

Les vétérans ont également un ratio de victoires aux points élevé, mais ils se distinguent surtout par leur volume important de combats aux points.

Les vulnérables aux points ont un ratio de victoires aux points moyen, tandis que les peu orientés points ont logiquement le ratio le plus faible. Cela confirme que les peu orientés points ne gagnent pas souvent par décision.

% Défaites aux points

Les vulnérables aux points se distinguent nettement par le ratio de défaites aux points le plus élevé. C’est leur caractéristique principale : lorsque le combat se décide aux points, ils sont plus souvent en difficulté.

À l’inverse, les efficaces aux points ont un ratio de défaites aux points faible, ce qui renforce l’idée qu’ils sont performants dans ce type de situation.

Les peu orientés points ont aussi un ratio de défaites aux points faible, mais cela s’explique surtout par le fait que leurs combats se terminent moins souvent aux points.

Nombre total de combats aux points

Les vétérans sont clairement séparés des autres groupes par un nombre beaucoup plus élevé de combats terminés aux points. Cela confirme que leur profil est principalement lié à l’expérience et au volume de combats.

Les efficaces aux points ont un volume plus modéré : ils ne vont pas forcément aussi souvent aux points que les vétérans, mais ils y sont plus rentables.

Les peu orientés points ont le plus faible volume de combats aux points, ce qui confirme que ce mode de victoire n’est pas central dans leur profil.

% Victoires global

Les efficaces aux points présentent le meilleur taux de victoires global. Cela suggère que leur capacité à bien gérer les combats aux points est associée à une bonne performance générale.

Les vétérans ont également un bon taux de victoires global, ce qui est cohérent avec leur expérience.

Les vulnérables aux points ont le taux de victoires global le plus faible, ce qui montre que leur fragilité aux points est associée à une performance générale plus faible.

Les peu orientés points restent compétitifs globalement, ce qui montre qu’un combattant peut être performant sans gagner souvent aux points, probablement en s’appuyant sur d’autres modes de victoire comme les soumissions.

In [ ]:
pca = PCA(n_components=2)
X_pca_points = pca.fit_transform(X_points_scaled)

labels = {
    0: "Peu orientés points",
    1: "Vulnérables aux points",
    2: "Vétérans",
    3: "Efficace aux points"
}

colors = {
    0: "#e74c3c",
    1: "#2ecc71",
    2: "#f39c12",
    3: "#3498db" 
}

fig, ax = plt.subplots(figsize=(10, 7))

for cluster in sorted(features_points["cluster_points"].unique()):
    idx = features_points["cluster_points"] == cluster
    
    ax.scatter(
        X_pca_points[idx, 0],
        X_pca_points[idx, 1],
        color=colors[cluster],
        label=labels[cluster],
        alpha=0.7,
        s=60,
        edgecolor="white",
        linewidth=0.4
    )

ax.set_xlabel(f"PC1 volume et expérience aux points ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 vulnérabilité aux points({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("PCA des profils aux points", fontsize=13, fontweight="bold")

ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Variables utilisées pour la PCA points
variables_points = [
    "ratio_off_point",
    "ratio_def_point",
    "total_points",
    "total_combats"
]

# Loadings : importance/signature de chaque variable dans PC1 et PC2
pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=variables_points,
    columns=["PC1", "PC2"]
)

print("Loadings PCA :")
print(pca_loadings.round(2))

print("\nVariance expliquée :")
print(pd.Series(
    pca.explained_variance_ratio_,
    index=["PC1", "PC2"]
).round(3))

PC1 (axe horizontal : affinité avec les victoires aux points)

La séparation de gauche à droite oppose principalement les combattants peu orientés vers les points aux combattants qui gagnent davantage aux points ou qui ont plus souvent des combats décidés aux points.

Les peu orientés points (rouge) sont majoritairement situés à gauche : ils ont peu de victoires aux points et peu de combats terminés aux points. Cela suggère que leurs combats se terminent davantage par d’autres issues, comme les soumissions.

Les vétérans (orange) sont plutôt situés à droite : ils ont un volume plus élevé de combats terminés aux points, ce qui traduit une forte présence dans ce type de décision. Cela est cohérent avec leur expérience et leur nombre total de combats plus important.

Les efficaces aux points (bleu) sont également situés plutôt du côté droit, mais davantage vers le bas : ils gagnent souvent aux points tout en ayant une vulnérabilité plus faible aux défaites aux points.

PC2 (axe vertical : vulnérabilité aux points)

Le deuxième axe sépare principalement les combattants selon leur vulnérabilité aux points.

Les vulnérables aux points (vert) sont clairement situés vers le haut : ils ont un ratio de défaites aux points plus élevé que les autres groupes. Ce cluster correspond donc aux combattants les plus fragiles lorsque le combat se décide aux points.

À l’inverse, les efficaces aux points (bleu) et les peu orientés points (rouge) sont plutôt situés vers le bas, ce qui indique une plus faible proportion de défaites aux points. Pour les peu orientés points, cela s’explique surtout par le fait que leurs combats se terminent moins souvent aux points.

La PCA confirme donc que les profils aux points sont structurés autour de deux dimensions principales : d’une part l’affinité avec les victoires ou les décisions aux points, et d’autre part la vulnérabilité lorsque le combat se décide aux points.

Comparaison avec le cluster précédent

In [ ]:
features_points["cluster"] = features["cluster"]

features_points.groupby("cluster")[[
    "ratio_off_point",
    "ratio_def_point",
    "total_points",
    "total_combats"
]].mean().round(2)

In [ ]:
colors = {0: '#2ecc71', 1: '#3498db', 2: '#e74c3c', 3: '#f39c12'}

labels = {
    0: "Vulnérables",
    1: "Finishers",
    2: "Techniciens",
    3: "Vétérans"
}

fig, ax = plt.subplots(figsize=(12, 8))

for cluster in range(4):
    data = features_points[features_points["cluster"] == cluster]
    ax.scatter(
        data["ratio_off_point"],
        data["ratio_def_point"],
        color=colors[cluster],
        label=labels[cluster],
        alpha=0.7,
        s=60
    )

ax.axvline(
    x=features_points["ratio_off_point"].mean(),
    color="gray",
    linestyle="--",
    alpha=0.5
)

ax.axhline(
    y=features_points["ratio_def_point"].mean(),
    color="gray",
    linestyle="--",
    alpha=0.5
)

ax.set_xlabel("% victoires aux points", fontsize=12)
ax.set_ylabel("% défaites aux points", fontsize=12)

ax.set_title(
    "Profil victoires/défaites aux points selon le clustering précédent",
    fontsize=13,
    fontweight="bold"
)

ax.legend(fontsize=11)


plt.tight_layout()
plt.show()

En projetant les clusters construits à partir des soumissions sur les variables liées aux points, on observe que les profils ne se comportent pas tous de la même manière.

Les finishers sont davantage présents dans la partie basse du graphique. Ils ont donc tendance à perdre relativement peu aux points. En revanche, ils ne sont pas particulièrement concentrés dans la zone des fortes victoires aux points, ce qui confirme qu’ils sont surtout orientés vers la finalisation par soumission plutôt que vers les décisions aux points.

Les vulnérables sont plus dispersés sur le graphique. Ils ne se distinguent pas aussi clairement sur les variables liées aux points que sur celles liées aux soumissions. Leur fragilité principale semble donc davantage concerner les défaites par soumission que les combats se terminant aux points.

Les techniciens se situent plutôt à droite du graphique, ce qui indique une tendance plus forte à gagner aux points. Cela complète leur profil observé précédemment : ils gagnent et perdent relativement peu par soumission, mais semblent davantage capables de remporter des combats aux points. On peut donc les interpréter comme des profils plus contrôlés, privilégiant la gestion du combat plutôt que la recherche systématique de la soumission.

Les vétérans sont également assez présents dans la partie droite du graphique, ce qui suggère qu’ils ont souvent remporté des combats aux points. Cela peut s’expliquer par leur grand volume de combats : plus un combattant possède d’expérience, plus il a de chances d’avoir accumulé différents types de victoires, notamment aux points.

Cependant, il faut rester prudent dans l’interprétation : ce graphique ne permet pas d’affirmer que les vétérans gagnent davantage aux points à cause de leur expérience, mais seulement qu’ils sont davantage représentés dans cette zone du graphique.

In [ ]:

pca = PCA(n_components=2)
X_pca_points = pca.fit_transform(X_points_scaled)

fig, ax = plt.subplots(figsize=(10, 7))

for cluster in range(4):
    idx = features_points["cluster"] == cluster
    ax.scatter(
        X_pca_points[idx, 0],
        X_pca_points[idx, 1],
        color=colors[cluster],
        label=labels[cluster],
        alpha=0.7,
        s=60
    )

ax.set_xlabel(f"PC1 volume et expérience aux points ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 vulnérabilité aux points({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("PCA des profils aux points", fontsize=13, fontweight="bold")

ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

Cette PCA permet de comparer les profils construits à partir des soumissions avec leur comportement dans les combats décidés aux points.

Les finishers se situent plutôt à gauche et en bas du graphique, ce qui confirme qu’ils ont une faible affinité avec les victoires aux points et une faible vulnérabilité aux points. Cela renforce l’idée qu’ils sont principalement orientés vers la finalisation par soumission plutôt que vers les décisions aux points.

Les vétérans sont davantage situés vers la droite du graphique, ce qui traduit une plus forte affinité avec les victoires aux points. Cette observation est cohérente avec leur grand nombre de combats : leur expérience les amène probablement à avoir davantage de combats allant à la décision.

Les techniciens sont plus hétérogènes. Une partie d’entre eux se rapproche des vétérans, avec une certaine affinité pour les victoires aux points, tandis qu’une autre partie reste plus centrale. Cela confirme qu’ils constituent un profil contrôlé et prudent face aux soumissions, mais pas nécessairement homogène dans leur manière de gagner aux points.

Enfin, les vulnérables ne se distinguent pas aussi clairement sur cette PCA des points. Leur dispersion suggère que leur fragilité principale concerne surtout les défaites par soumission, plutôt qu’un comportement spécifique face aux victoires ou défaites aux points.

In [ ]:
# Copier pour éviter de modifier features_points directement
features_points_soum = features_points.copy()

# Ajouter le cluster de soumissions
features_points_soum["cluster_soumission"] = features["cluster"].reindex(features_points_soum.index)

# Ajouter le ratio de victoires global si besoin
features_points_soum["ratio_victoires"] = corr_features["ratio_victoires"].reindex(features_points_soum.index)

# Supprimer les lignes sans cluster de soumission
features_points_soum = features_points_soum.dropna(subset=["cluster_soumission", "ratio_victoires"])

# Remettre le cluster en entier
features_points_soum["cluster_soumission"] = features_points_soum["cluster_soumission"].astype(int)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_names = [
    "ratio_off_point",
    "ratio_def_point",
    "total_points",
    "ratio_victoires"
]

titles = [
    "% Victoires aux points",
    "% Défaites aux points",
    "Nombre total de combats aux points",
    "% Victoires global"
]

colors_list = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12"]

labels = [
    "Vulnérables",
    "Finishers",
    "Techniciens",
    "Vétérans"
]

for ax, feature, title in zip(axes.flatten(), features_names, titles):
    data = [
        features_points_soum[
            features_points_soum["cluster_soumission"] == k
        ][feature].dropna().values
        for k in range(4)
    ]

    bp = ax.boxplot(data, patch_artist=True, tick_labels=labels)

    for patch, color in zip(bp["boxes"], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(title, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", rotation=15)

plt.suptitle(
    "Distribution des variables points selon les clusters de soumissions",
    fontsize=14,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
plt.show()

% Victoires aux points

Les techniciens et les vétérans sont les deux groupes qui gagnent le plus souvent aux points. Cela confirme que les techniciens, qui étaient moins orientés vers les soumissions, compensent davantage par des victoires aux points. Les vétérans présentent aussi un ratio élevé, probablement lié à leur expérience et à leur volume important de combats.

À l’inverse, les finishers ont un ratio de victoires aux points plus faible. Cela est cohérent avec leur profil : ils privilégient la finalisation par soumission plutôt que les décisions aux points.

% Défaites aux points

Les vulnérables et les techniciens présentent les ratios de défaites aux points les plus élevés. Pour les vulnérables, cela confirme une certaine fragilité globale : ils étaient déjà vulnérables aux soumissions et semblent aussi souvent en difficulté lorsque le combat se décide aux points.

Pour les techniciens, l’interprétation est différente : ils gagnent souvent aux points, mais ils perdent aussi plus souvent aux points que les finishers. Cela suggère qu’ils disputent davantage de combats allant à la décision, ce qui augmente à la fois leurs victoires et leurs défaites aux points.

Nombre total de combats aux points

Les vétérans se distinguent très nettement par un nombre beaucoup plus élevé de combats terminés aux points. Cela confirme que leur caractéristique principale reste le volume et l’expérience.

Les techniciens ont également un volume de combats aux points plus élevé que les finishers et les vulnérables, ce qui renforce l’idée qu’ils sont davantage associés à des combats contrôlés et décidés aux points.

Les finishers ont un volume plus faible de combats aux points, ce qui est cohérent avec leur style : leurs combats se terminent plus souvent par soumission que par décision.

Cette comparaison confirme que les profils construits à partir des soumissions se prolongent partiellement dans l’analyse des points : les finishers restent peu orientés vers les décisions, tandis que les techniciens et les vétérans sont davantage associés aux victoires aux points.

## Comparaison clusters points avec l'analyse soumissions

In [ ]:
features["cluster_points"] = features_points["cluster_points"]
features

In [ ]:
features.groupby("cluster_points")[[
    "ratio_offense",
    "ratio_defense",
    "total_combats"
]].mean().round(2)

In [ ]:
labels = {
    0: "Peu orientés points",
    1: "Vulnérables aux points",
    2: "Vétérans",
    3: "Efficace aux points"
}

colors = {
    0: "#e74c3c",
    1: "#2ecc71",
    2: "#f39c12",
    3: "#3498db" 
}
fig, ax = plt.subplots(figsize=(12, 8))

for cluster in range(4):
    data = features[features['cluster_points'] == cluster]
    ax.scatter(data['ratio_offense'], data['ratio_defense'],
               color=colors[cluster], label=labels[cluster],
               alpha=0.7, s=60)

ax.axvline(x=features['ratio_offense'].mean(), color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=features['ratio_defense'].mean(), color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('% victoires par soumission (offensive)', fontsize=12)
ax.set_ylabel('% défaites par soumission (vulnérabilité)', fontsize=12)
ax.set_title('Comparaison du clustering points appliqué à nos données de soumissions', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

En projetant les clusters construits à partir des points sur les variables de soumission, on observe que les groupes ne sont pas parfaitement séparés. Cela montre que les profils aux points et les profils aux soumissions ne décrivent pas exactement la même dimension du style de combat.

Les combattants peu orientés points se situent davantage à droite du graphique, ce qui indique un ratio de victoires par soumission plus élevé. Cela est cohérent : s’ils gagnent peu aux points, c’est souvent parce qu’ils obtiennent davantage de victoires par soumission.

Les vétérans sont plutôt regroupés dans la partie basse/centrale : ils gagnent moins souvent par soumission et subissent peu de défaites par soumission. Cela confirme un profil plus contrôlé, orienté vers les décisions plutôt que vers la finalisation.

Les efficaces aux points ont un profil plus intermédiaire : ils ne sont pas les plus orientés soumissions, mais ils restent globalement solides défensivement.

Les vulnérables aux points sont assez dispersés. Leur fragilité aux points ne signifie donc pas forcément qu’ils sont aussi vulnérables aux soumissions.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

labels = {
    0: "Peu orientés points",
    1: "Vulnérables aux points",
    2: "Vétérans",
    3: "Efficace aux points"
}

colors = {
    0: "#e74c3c",
    1: "#2ecc71",
    2: "#f39c12",
    3: "#3498db" 
}

for cluster in range(4):
    mask = features['cluster_points'] == cluster
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=colors[cluster], label=labels[cluster],
               alpha=0.7, s=60)

plt.xlabel("PC1 : Profil de performance / style", fontsize = 12)
ax.set_ylabel('PC2 : Expérience (24.7%)', fontsize=12)
ax.set_title('PCA de notre comparaison du clustering points', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

La séparation semble ici principalement se faire sur le deuxième axe (PC2), qui correspond davantage à l’expérience et au volume de combats.

Les vétérans sont clairement situés dans la partie haute du graphique, ce qui confirme qu’ils possèdent un nombre de combats plus élevé et donc une plus grande expérience.

À l’inverse, les profils peu orientés points sont davantage situés dans la partie basse de PC2. Cela suggère qu’ils correspondent à des combattants plus récents ou moins expérimentés, dont les combats se terminent moins souvent aux points.

Cette observation est cohérente avec l’évolution récente du jiu-jitsu brésilien, où certaines compétitions modernes semblent davantage valoriser les styles agressifs et les victoires par soumission. Les combattants les plus récents pourraient donc être moins orientés vers les décisions aux points que les profils plus anciens et expérimentés.

Cependant, cette interprétation reste une hypothèse issue des tendances observées dans les données et ne constitue pas une preuve directe d’une évolution du sport.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_names = ['ratio_offense', 'ratio_defense', 'total_combats', 'ratio_victoires']
titles = ['% Victoires par soumission (offense)', 
          '% Défaites par soumission (defense)',
          'Nombre total de combats',
          '% Victoires global']

colors_list = ["#e74c3c", "#2ecc71","#f39c12" , "#3498db"]

labels = [
    "Peu orientés\npoints",
    "Vulnérables\naux points",
    "Vétérans",
    "Efficaces\naux points"
    
]
for ax, feature, title in zip(axes.flatten(), features_names, titles):
    data = [features[features['cluster_points'] == k][feature].values for k in range(4)]
    bp = ax.boxplot(data, patch_artist=True, tick_labels=labels)
    
    for patch, color in zip(bp['boxes'], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Distribution des features par cluster', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

% Victoires par soumission

Les peu orientés points se démarquent nettement avec le ratio de victoires par soumission le plus élevé. Cela confirme qu’ils ne sont pas peu performants : ils sont simplement moins orientés vers les décisions aux points et gagnent davantage par soumission.

Les vétérans ont également un ratio de victoires par soumission correct, mais leur caractéristique principale reste surtout leur nombre total de combats plus élevé.

Les vulnérables aux points et les efficaces aux points ont des ratios de victoires par soumission plus modérés. Cela suggère que leur profil est moins centré sur la finalisation par soumission.

% Défaites par soumission

Les vétérans et les efficaces aux points présentent les ratios de défaites par soumission les plus faibles. Cela indique qu’ils sont globalement plus solides face aux soumissions.

Les peu orientés points ont un ratio de défaites par soumission plus élevé que les vétérans et les efficaces aux points. Cela peut traduire un style plus offensif et plus risqué : ils gagnent davantage par soumission, mais peuvent aussi s’exposer davantage.

Les vulnérables aux points restent assez dispersés. Leur vulnérabilité principale concerne surtout les décisions aux points, pas nécessairement les soumissions.

Nombre total de combats

Les vétérans se distinguent très nettement par un nombre total de combats beaucoup plus élevé. Cela confirme que ce cluster est principalement lié à l’expérience et au volume de combats.

Les autres groupes ont un nombre de combats plus faible et relativement proche, même si les peu orientés points présentent une certaine dispersion.

% Victoires global

Les efficaces aux points et les vétérans présentent les meilleurs taux de victoires globaux. Cela suggère que l’efficacité aux points et l’expérience sont associées à une bonne performance générale.

Les peu orientés points restent compétitifs, notamment grâce à leur forte capacité à gagner par soumission, mais leur profil semble plus variable.

Les vulnérables aux points ont le taux de victoires global le plus faible, ce qui confirme que leur fragilité aux points est associée à une performance générale plus faible.

In [ ]:
def affichage_repartion(features, g1, g2, title, group_names=None, value_names=None):
    groups = features.groupby(g1)[g2]
    repartition_groupe = []
    
    for i in range(4):
        groupe_i = groups.get_group(i)
        repartition_groupe.append(groupe_i.value_counts().sort_index())
    
    colors_list = ["#e74c3c", "#2ecc71", "#f39c12", "#3498db"]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), squeeze=False)
    
    for i in range(4):
        ax = axes[i // 2, i % 2]
        data = repartition_groupe[i]
        
        xlabels = [value_names[int(j)] for j in data.index] if value_names else data.index
        cluster_title = group_names[int(i)] if group_names else f"Cluster {i}"
        colors = [colors_list[int(j)] for j in data.index]
        
        ax.pie(data.values, labels=xlabels, autopct="%1.1f%%", colors=colors)
        ax.set_title(cluster_title, fontsize=13, fontweight="bold")
    
    plt.suptitle(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
cluster_names = {0: "Vulnérables", 1: "Finishers", 2: "Techniciens", 3: "Vétérans"}
point_names = {0: "Peu orientés points", 1: "Vulnérables aux points", 2: "Vétérans", 3: "Efficace aux points"}

affichage_repartion(features, "cluster", "cluster_points",
                    title="Répartition des cluster points par cluster soumissions",
                    group_names=cluster_names,
                    value_names=point_names)

Ces camemberts permettent de croiser nos deux analyses et d'en tirer de nouvelles conclusions.

Vulnérables aux soumissions :
Comme attendu, ils sont majoritairement vulnérables aux points également (52%). La nouvelle information est qu'un tiers d'entre eux sont peu orientés points, ce qui suggère qu'ils cherchent eux-mêmes à gagner par soumissions sans pour autant y parvenir. On se retrouve alors dans le pire des cas : une défense aux soumissions fragile et peu d'efficacité offensive. La priorité pour ce profil est claire : renforcer la défense ou développer un jeu de points solide.

Finishers :
Sans surprise, ils sont très peu orientés vers les victoires aux points (63.6%), ce qui s'explique naturellement : leurs combats se terminent souvent avant le verdict des juges. Ce qui est plus intéressant est que 22% d'entre eux sont également efficaces aux points, un profil polyvalent et difficile à contrer. C'est la catégorie vers laquelle tout compétiteur devrait tendre.

Techniciens :
C'est le cluster le plus hétérogène. Ne pas perdre par soumissions n'implique pas de gagner par points. On trouve autant de profils efficaces aux points (40%) que vulnérables aux points (40%). C'est aussi le cluster qui présente le plus de vétérans (10.2%), ce qui confirme que la défense est une base solide pour la longévité. Cependant, sans capacité à marquer des points ou à soumettre, la défense seule ne suffit pas pour gagner.

Vétérans :
Profil très lisible. 92.4% sont classés vétérans en points également. Cela confirme une conclusion forte : la longévité en compétition est fortement corrélée à la maîtrise du jeu de points. Les vétérans ne se définissent pas uniquement par leur expérience mais par leur capacité à contrôler un combat sur la durée.

Le croisement des deux clusterings révèle que la défense aux soumissions et la maîtrise des points sont deux compétences largement indépendantes. Les profils les plus complets, et les plus difficiles à battre, sont ceux qui maîtrisent les deux dimensions à la fois.

In [ ]:
affichage_repartion(features_points, "cluster_points", "cluster",
                    title="Répartition des cluster soumissions par cluster point",
                    group_names=point_names,
                    value_names=cluster_names)

Peu orientés points :
Les combattants peu orientés points sont majoritairement des Finishers (56.6%), ce qui est cohérent. Ne pas chercher les points implique de chercher la soumission. Cependant, 27.5% restent vulnérables aux soumissions, ce qui n'est pas une coïncidence : un combattant qui cherche à soumettre s'expose lui-même davantage aux contre-attaques. C'est le même profil offensif qui génère la vulnérabilité défensive.

Vulnérables aux points :
La majorité des vulnérables aux points sont des Techniciens (49%), ce qui confirme une conclusion importante : savoir se défendre des soumissions ne suffit pas à gagner des points. Le profil Technicien est donc potentiellement frustrant, solide défensivement mais sans réelle menace offensive, ce qui le rend prévisible et difficile à valoriser en compétition.

Vétérans :
Sans surprise, les vétérans restent majoritairement vétérans (76.2%). Ce qui est plus intéressant est que 16.8% sont des Techniciens. L'expérience accumulée permet donc de développer une résistance aux soumissions, même sans être naturellement offensif. C'est une progression logique de carrière : on apprend d'abord à ne plus se faire soumettre.

Efficaces aux points :
Les efficaces aux points sont très majoritairement des Techniciens (62.7%). La logique est claire, pour accumuler des points il faut tenir la durée du combat, ce qui implique de résister aux soumissions adverses. On retrouve également 24% de Finishers, confirmant que les profils les plus complets combinent maîtrise des points et efficacité aux soumissions : les adversaires les plus difficiles à battre.

Ce second croisement renforce la conclusion précédente : la résistance aux soumissions est une condition nécessaire mais pas suffisante pour performer. Les profils dominants sont ceux qui y ajoutent soit une efficacité offensive par soumissions, soit une maîtrise du jeu de points.

# Analyse de l'évolution des clustering

In [ ]:
def compute_features_year(df, year, min_combats=5,cumulatif = True):
    if cumulatif :
        df_year = df[df["Annee"] <= year]  # cumulatif
    else :
        df_year = df[df["Annee"] == year] 
    total_combats = df_year.groupby("id_combattant").size()
    
    # Défaites par soumission
    defaites_year = df_year[df_year["Résultat"] == "L"].copy()
    defaites_year["categorie"] = defaites_year["Type_victoire"].apply(categoriser)
    defaites_soum_year = defaites_year[defaites_year["categorie"].isin(soumissions_off)]
    def_soum = defaites_soum_year.groupby("id_combattant").size()
    ratio_soum = (def_soum / total_combats * 100)
    
    # Victoires par soumission
    victoires_year = df_year[df_year["Résultat"] == "W"].copy()
    victoires_year["categorie"] = victoires_year["Type_victoire"].apply(categoriser)
    victoires_soum_year = victoires_year[victoires_year["categorie"].isin(soumissions_off)]
    vic_soum = victoires_soum_year.groupby("id_combattant").size()
    ratio_off = (vic_soum / total_combats * 100)
    
    features_year = pd.DataFrame({
        "ratio_offense": ratio_off,
        "ratio_defense": ratio_soum,
        "total_combats": total_combats,
        "ratio_victoires": df_year.groupby("id_combattant")["Résultat"].apply(
            lambda x: (x == "W").sum() / len(x) * 100
        )
    }).dropna()
    
    features_year = features_year[features_year["total_combats"] >= min_combats]
    return features_year

In [ ]:
def creation_trajectoire(cumulatif = True ):
    trajectoires = {}
    
    FEATURES_COLS = ["ratio_offense", "ratio_defense", "total_combats", "ratio_victoires"]
    
    scaler_soum = StandardScaler()
    X_soum = scaler_soum.fit_transform(features[FEATURES_COLS])
    
    kmeans_soum = KMeans(n_clusters=4, random_state=42, n_init=10)
    kmeans_soum.fit(X_soum)
    
    # Puis dans la boucle temporelle
    for year in range(2010, 2026):
        features_year = compute_features_year(df, year, min_combats=5,cumulatif = cumulatif)
        X = scaler_soum.transform(features_year[FEATURES_COLS])
        features_year["cluster"] = kmeans_soum.predict(X)    
        for fighter_id, row in features_year.iterrows():
            if fighter_id not in trajectoires:
                trajectoires[fighter_id] = {}
            trajectoires[fighter_id][year] = row["cluster"]
    return trajectoires

In [ ]:
def matrice_transition(trajectoires,cluster_names = cluster_names):
    transitions = np.zeros((4, 4))
    
    for fighter, years in trajectoires.items():
        sorted_years = sorted(years.keys())
        for i in range(len(sorted_years) - 1):
            c_before = int(years[sorted_years[i]])
            c_after = int(years[sorted_years[i + 1]])
            transitions[c_before][c_after] += 1
    
    transitions_pct = transitions / transitions.sum(axis=1, keepdims=True) * 100
    return pd.DataFrame(transitions_pct, index=cluster_names.values(), columns=cluster_names.values()).round(1)

In [ ]:
def plot_volume(trajectoires,cluster_names = cluster_names):
    # Compter le nombre de combattants par cluster par année
    volume_par_annee = {i: {} for i in range(4)}
    
    for fighter, years in trajectoires.items():
        for year, cluster in years.items():
            cluster = int(cluster)
            if year not in volume_par_annee[cluster]:
                volume_par_annee[cluster][year] = 0
            volume_par_annee[cluster][year] += 1
    
    years_list = sorted(range(2013, 2026))
    colors_list = ["#2ecc71","#3498db" , "#e74c3c", "#f39c12"]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for i in range(4):
        volumes = [volume_par_annee[i].get(y, 0) for y in years_list]
        ax.plot(years_list, volumes, marker="o", label=cluster_names[i], 
                color=colors_list[i], linewidth=2)
        ax.fill_between(years_list, volumes, alpha=0.1, color=colors_list[i])
    
    ax.set_title("Volume de combattants par cluster par année", fontsize=14, fontweight="bold")
    ax.set_xlabel("Année")
    ax.set_ylabel("Nombre de combattants")
    ax.legend()
    ax.set_xticks(years_list)
    plt.tight_layout()
    plt.show()

## Analyse par cumulation des années

L'analyse cumulative consiste à recalculer les features de chaque combattant en intégrant l'ensemble de ses combats jusqu'à une année donnée. Cette approche présente l'avantage de capturer le profil structurel d'un combattant, celui qui émerge sur l'ensemble de sa carrière. Les ratios étant construits sur un grand nombre de combats, ils sont stables et peu sensibles aux performances ponctuelles. C'est donc l'approche la plus adaptée pour identifier des typologies durables et comprendre comment un combattant se positionne globalement dans la compétition. En revanche, cette stabilité est aussi sa limite : plus un combattant accumule de combats, plus son cluster "gèle" et les transitions deviennent rares, ce qui rend difficile la détection d'une évolution réelle de style en cours de carrière.

In [ ]:
trajectoires_cumulatif = creation_trajectoire()
matrice_transition(trajectoires_cumulatif)

La matrice de transition cumulative confirme ce que l'approche théorique laissait anticiper : les clusters sont très stables dans le temps. Les diagonales dominent largement avec 88.5% pour les Vulnérables, 86.8% pour les Finishers, 90.3% pour les Techniciens et 99.8% pour les Vétérans. Ces chiffres signifient qu'une fois qu'un combattant est assigné à un profil, il y reste presque systématiquement d'une année à l'autre. Cela tient directement à la nature cumulative de l'analyse, les ratios étant construits sur l'ensemble de la carrière, l'ajout d'une année supplémentaire modifie marginalement les features et donc rarement le cluster attribué.
Malgré cette stabilité, quelques transitions méritent attention. Le chemin le plus fréquent pour les Vulnérables est de devenir Techniciens (9.1%), ce qui suggère qu'avec l'expérience, certains combattants apprennent à corriger leur défense sans pour autant développer une offensive par soumissions. Les Finishers quant à eux transitent principalement vers les Techniciens (7.6%) ou les Vétérans (3.9%), ce qui traduit une évolution naturelle vers des profils plus équilibrés et expérimentés en fin de carrière. Enfin, le cas des Vétérans est sans appel, 99.8% de stabilité confirme que ce profil est quasi irréversible : une fois acquise, l'expérience accumulée ne se perd pas.

In [ ]:
for i, name in cluster_names.items():
    fighters_in_cluster = [f for f, y in trajectoires_cumulatif.items() if i in y.values()]
    stable = [f for f in fighters_in_cluster 
              if all(c == i for c in trajectoires_cumulatif[f].values())]
    print(f"{name}: {len(stable)}/{len(fighters_in_cluster)} stables ({len(stable)/len(fighters_in_cluster)*100:.1f}%)")

Les Finishers sont le profil le plus stable (43.3%), suivis des Techniciens (37.9%) et des Vulnérables (35.8%). Les Vétérans affichent paradoxalement la stabilité la plus faible (1.7%), ce qui s'explique par le fait qu'ils n'apparaissent dans ce cluster qu'en fin de carrière, ils ont donc nécessairement transité par d'autres profils auparavant.

In [ ]:
plot_volume(trajectoires_cumulatif)

Les Techniciens dominent et croissent le plus rapidement, ce qui suggère que le niveau général en BJJ progresse : de plus en plus de combattants développent une défense solide aux soumissions. Les Vétérans apparaissent tardivement, quasi absents avant 2016, et progressent régulièrement, logique puisqu'il faut accumuler suffisamment de combats pour atteindre ce profil. Vulnérables et Finishers progressent de façon similaire et parallèle, ce qui confirme leur lien structurel observé dans les camemberts précédents.

## Analyse annuelle

L'analyse annuelle consiste à ne considérer que les combats réalisés durant une année donnée. Cette approche est plus sensible aux changements de style et permet de capturer la dynamique de carrière d'un combattant, une progression, un déclin, ou une reconversion tactique. Elle est particulièrement pertinente pour les combattants actifs qui cherchent à faire évoluer leur jeu. Sa principale limite est la sensibilité au faible volume de données : un combattant qui ne dispute que 1 à 2 combats dans l'année aura des ratios peu fiables, ce qui peut générer des classifications erronées. C'est pourquoi un seuil minimum de combats par année est nécessaire, au prix d'une perte de couverture sur certains profils comme les Vétérans qui combattent moins fréquemment.

In [ ]:
trajectoires_annee = creation_trajectoire(cumulatif=False)
matrice_transition(trajectoires_annee)

En annuel pur, les transitions sont beaucoup plus fréquentes et les diagonales nettement plus faibles qu'en cumulatif. Les Finishers restent le profil le plus stable (48.1%) tandis que les Techniciens et Vulnérables changent fréquemment de cluster d'une année à l'autre, avec respectivement 48.1% et 54% de stabilité seulement. Cela confirme que ces profils sont plus sensibles aux variations de performance annuelle.

Les transitions les plus notables sont les Vulnérables qui deviennent Techniciens (29.1%) et les Techniciens qui deviennent Vulnérables (32%), ce qui traduit une forte porosité entre ces deux profils, un combattant peut osciller entre les deux selon ses performances de l'année. Les Vétérans disparaissent complètement en annuel pur, ce qui était attendu : leur profil repose sur l'accumulation de combats et non sur une performance annuelle isolée.

La stabilité individuelle confirme cette tendance : seulement 26.6% des Vulnérables, 27% des Finishers et 20.6% des Techniciens restent dans leur cluster sur toute la période, contre des taux bien plus élevés en cumulatif. L'analyse annuelle révèle donc une réalité plus mouvante où les profils ne sont pas figés et où un combattant peut significativement évoluer d'une saison à l'autre.

In [ ]:
for i, name in cluster_names.items():
    fighters_in_cluster = [f for f, y in trajectoires_annee.items() if i in y.values()]
    stable = [f for f in fighters_in_cluster 
              if all(c == i for c in trajectoires_annee[f].values())]
    if name!="Vétérans":
        print(f"{name}: {len(stable)}/{len(fighters_in_cluster)} stables ({len(stable)/len(fighters_in_cluster)*100:.1f}%)")

In [ ]:
transitions = np.zeros((4, 4))
    
# Transitions normales en annuel pur pour clusters 0, 1, 2
for fighter, years in trajectoires_annee.items():
    sorted_years = sorted(years.keys())
    for i in range(len(sorted_years) - 1):
        c_before = int(years[sorted_years[i]])
        c_after = int(years[sorted_years[i + 1]])
        if c_before != 3:  # ignorer la ligne vétérans
            transitions[c_before][c_after] += 1

# Ligne vétérans : depuis cumulatif vers annuel pur
veterans_ids = [f for f, years in trajectoires_cumulatif.items()
                if int(years[max(years.keys())]) == 3]

for fighter_id in veterans_ids:
    if fighter_id in trajectoires_annee:
        last_year = max(trajectoires_annee[fighter_id].keys())
        c_after = int(trajectoires_annee[fighter_id][last_year])
        transitions[3][c_after] += 1
    
transitions_pct = transitions / transitions.sum(axis=1, keepdims=True) * 100
pd.DataFrame(transitions_pct, index=cluster_names.values(), columns=cluster_names.values()).round(1).fillna(0)

In [ ]:
plot_volume(trajectoires_annee)

Contrairement au cumulatif, le volume annuel est beaucoup plus volatile et révèle des dynamiques intéressantes. La chute visible en 2020 sur tous les clusters s'explique très probablement par le Covid qui a fortement réduit le nombre de compétitions cette année-là. Les trois profils principaux progressent globalement sur la période mais avec des fluctuations importantes, ce qui confirme que les classifications annuelles sont sensibles au calendrier compétitif. Les Vulnérables dominent depuis 2022, ce qui peut suggérer un afflux de nouveaux compétiteurs moins expérimentés attirés par la croissance du BJJ. Les Vétérans restent quasi absents comme attendu, le seuil minimum de combats par année étant trop contraignant pour ce profil

Refaisons la même analyse mais pour le second clustering

## Cumaltif points

In [ ]:
def compute_features_year_points(df, year, min_combats=5, cumulatif=True):
    if cumulatif:
        df_year = df[df["Annee"] <= year]
    else:
        df_year = df[df["Annee"] == year]
    
    df_year = df_year.copy()
    df_year["categorie"] = df_year["Type_victoire"].apply(categoriser)
    df_year["Résultat_bool"] = df_year["Résultat"] == "W"
    
    total_combats = df_year.groupby("id_combattant").size()
    
    # Victoires par points
    victoires_point = df_year[df_year["Résultat_bool"] & (df_year["categorie"] == "Points/Avantages")]
    ratio_off_point = victoires_point.groupby("id_combattant").size() / total_combats * 100
    
    # Défaites par points
    defaites_point = df_year[~df_year["Résultat_bool"] & (df_year["categorie"] == "Points/Avantages")]
    ratio_def_point = defaites_point.groupby("id_combattant").size() / total_combats * 100
    
    # Total combats impliquant les points
    total_points = df_year[df_year["categorie"] == "Points/Avantages"].groupby("id_combattant").size()
    
    features_year = pd.DataFrame({
    "ratio_off_point": ratio_off_point,
    "ratio_def_point": ratio_def_point,
    "total_points": total_points,
    "total_combats": total_combats,  # ajouter ici
    }).dropna()
    
    features_year = features_year[features_year["total_combats"] >= min_combats]
    return features_year

In [ ]:
def creation_trajectoire_points(cumulatif=True):
    trajectoires = {}
    FEATURES_COLS = ["ratio_off_point", "ratio_def_point", "total_points", "total_combats"]
    
    scaler_pts = StandardScaler()
    X = scaler_pts.fit_transform(features_points[FEATURES_COLS])
    
    kmeans_pts = KMeans(n_clusters=4, random_state=42, n_init=10)
    kmeans_pts.fit(X)
    
    for year in range(2010, 2026):
        features_year = compute_features_year_points(df, year, min_combats=5, cumulatif=cumulatif)
        X_year = scaler_pts.transform(features_year[FEATURES_COLS])
        features_year["cluster"] = kmeans_pts.predict(X_year)
        
        for fighter_id, row in features_year.iterrows():
            if fighter_id not in trajectoires:
                trajectoires[fighter_id] = {}
            trajectoires[fighter_id][year] = row["cluster"]
    
    return trajectoires

In [ ]:
trajectoires_points_cumulatif = creation_trajectoire_points()
cluster_points_name = { 0 : "Peu orientées points", 1:"Vulnérables aux points", 2: "Vétérans", 3:"Efficaces aux points"} # nom au pif
matrice_transition(trajectoires_points_cumulatif,cluster_names= cluster_points_name)

Les clusters points sont encore plus stables que les clusters soumissions, avec des diagonales très élevées : 88% pour les Peu orientés points, 88.6% pour les Vulnérables aux points et 85.6% pour les Efficaces aux points. Les Vétérans atteignent 100%, ce qui confirme une fois de plus que ce profil est totalement irréversible en cumulatif.
Les transitions les plus notables sont les Peu orientés points qui deviennent Vulnérables aux points (5.9%) et les Efficaces aux points qui deviennent Vulnérables (6.1%), ce qui traduit une dégradation possible du profil offensif avec le temps — un combattant efficace aux points peut progressivement accumuler plus de défaites aux points en affrontant des adversaires de meilleur niveau. La transition inverse, Vulnérables vers Efficaces (4.4%), reste rare mais existe, suggérant qu'une progression est possible mais difficile.

In [ ]:
plot_volume(trajectoires_points_cumulatif,cluster_names=cluster_points_name)

Les Vulnérables aux points dominent largement et progressent le plus rapidement, ce qui confirme que la maîtrise du jeu de points reste le défi principal pour la majorité des compétiteurs. Les Efficaces aux points partaient pourtant avec un volume initial élevé en 2013 mais se font progressivement dépasser, suggérant que l'afflux de nouveaux compétiteurs moins expérimentés tire le niveau moyen vers le bas. Les Vétérans apparaissent tardivement et progressent régulièrement comme attendu. Fait notable, contrairement au clustering soumissions où les Techniciens dominaient, ici ce sont les Vulnérables aux points qui sont majoritaires, ce qui confirme que la maîtrise des points est globalement plus difficile à acquérir que la résistance aux soumissions.

## Année points

In [ ]:
trajectoires_points_annee = creation_trajectoire_points(cumulatif=False)
matrice_transition(trajectoires_points_annee,cluster_names= cluster_points_name)

Les transitions sont beaucoup plus fréquentes qu'en cumulatif, confirmant la plus grande volatilité de l'analyse annuelle.Les transitions sont beaucoup plus fréquentes qu'en cumulatif, confirmant la plus grande volatilité de l'analyse annuelle. Les Vulnérables aux points sont le profil le plus instable, seulement 47.8% restent dans ce cluster d'une année à l'autre, et 25.6% basculent vers les Peu orientés points, ce qui suggère qu'ils abandonnent le jeu de points plutôt que de l'améliorer. Les Efficaces aux points sont relativement stables (49.2%) mais 31.4% deviennent Vulnérables, traduisant la difficulté de maintenir un niveau offensif élevé sur la durée. Les Peu orientés points transitent fréquemment vers les Vulnérables (35.3%), ce qui est logique : en s'engageant davantage dans le jeu de points, on s'expose aussi à en perdre. Les Vétérans disparaissent comme attendu pour les mêmes raisons que dans le clustering soumissions.

In [ ]:
plot_volume(trajectoires_points_annee,cluster_names=cluster_points_name)

Le graphe annuel points révèle une dynamique très différente du cumulatif. Les Efficaces aux points dominent en début de période (2013-2019) avant d'être rattrapés puis dépassés par les Vulnérables et les Peu orientés points, ce qui confirme que la croissance du BJJ amène davantage de nouveaux compétiteurs peu à l'aise avec le jeu de points. La chute de 2020 est à nouveau visible sur tous les clusters, imputable au Covid. Ce qui est particulièrement intéressant est la convergence des trois profils principaux depuis 2022, suggérant une homogénéisation du niveau en termes de jeu de points. Les Vétérans restent absents comme attendu.